# w1-dict-features -- Compute Per-Review Dictionary Features

**Project:** Making Taste Legible: Symbolic Boundaries and Expert Valuation in Whisky Reviews

Compute dictionary-based features for every review across all 9 categories and 6 text scopes. Output a Parquet feature matrix joinable to the analytical dataset by `dedupe_hash`.

## 1. Load Data and Dictionary

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import os
import time
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

os.chdir('/Users/mac/Desktop/CSSS594/FinalProject')

# Load tokenized dataset
df = pd.read_parquet('data/whiskyfun_tokenized.parquet')
print(f"Loaded {len(df):,} reviews")
print(f"Text columns: {[c for c in df.columns if c not in ['dedupe_hash','score','review_year','distillery','review_length','identity_status','match_source','review_date','whisky_name_raw','source_url']]}")

# Load finalized dictionary
with open('data/whiskyfun_dictionary_v1.json') as f:
    dictionary = json.load(f)
print(f"\nLoaded dictionary v1: {dictionary['total_terms']} terms across {len(dictionary['categories'])} categories")

for cat, cdata in dictionary['categories'].items():
    print(f"  {cat}: {cdata['term_count']} terms")

## 2. Configuration: Scopes, Categories, and Feature Types

Define the text scopes, category short names, and feature types as specified.

In [ ]:
# Category short names for column prefixes
CAT_SHORT = {
    "fruit_aromatics": "fruit",
    "peat_smoke_coastal": "peat",
    "sherry_rancio_oxidative": "sherry",
    "oak_cask_wood": "oak",
    "texture_body": "texture",
    "mineral_earth_farmy": "mineral",
    "flaws_off_notes": "flaw",
    "complexity_balance": "structure",
    "explicit_evaluation": "eval",
}

# Text scopes
SCOPES = ["review_text", "nose", "mouth", "finish", "comments", "nmf"]

# Feature types
FEATURE_TYPES = ["count", "prop", "per1k", "binary"]

# Verify all scopes exist in dataframe
for scope in SCOPES:
    if scope in df.columns:
        non_null = df[scope].notna().sum()
        empty = df[scope].isna().sum() + (df[scope] == '').sum()
        print(f"  {scope}: {non_null:,} non-null, {empty:,} empty/missing")
    else:
        print(f"  WARNING: {scope} not in dataframe!")

print(f"\nTotal features to compute: {len(dictionary['categories'])} cats x {len(SCOPES)} scopes x {len(FEATURE_TYPES)} types = {len(dictionary['categories']) * len(SCOPES) * len(FEATURE_TYPES)}")

  review_text: 11,149 non-null, 0 empty/missing
  nose: 11,133 non-null, 16 empty/missing
  mouth: 11,130 non-null, 19 empty/missing
  finish: 11,120 non-null, 29 empty/missing
  comments: 11,100 non-null, 49 empty/missing
  nmf: 11,133 non-null, 16 empty/missing

Total features to compute: 9 cats x 6 scopes x 4 types = 216


## 3. Build Compiled Regex Patterns

For each category, compile a single regex pattern that matches any term with word-boundary matching. Terms are sorted by length descending to handle overlapping matches correctly.

In [ ]:
# Build patterns for each category
patterns = {}
cat_keys = list(dictionary['categories'].keys())

for cat_key in cat_keys:
    terms = dictionary['categories'][cat_key]['terms']
    # Sort by length descending to handle overlapping terms
    terms_sorted = sorted(terms, key=len, reverse=True)
    pattern = re.compile(
        r'\b(?:' + '|'.join(re.escape(t) for t in terms_sorted) + r')\b',
        re.IGNORECASE
    )
    patterns[cat_key] = pattern
    print(f"  {cat_key}: {len(terms)} terms, pattern length={len(pattern.pattern)}")

print(f"\nCompiled {len(patterns)} regex patterns.")

  fruit_aromatics: 42 terms, pattern length=358
  peat_smoke_coastal: 36 terms, pattern length=272
  sherry_rancio_oxidative: 33 terms, pattern length=269
  oak_cask_wood: 23 terms, pattern length=200
  texture_body: 30 terms, pattern length=190
  mineral_earth_farmy: 37 terms, pattern length=262
  flaws_off_notes: 27 terms, pattern length=229
  complexity_balance: 33 terms, pattern length=302
  explicit_evaluation: 20 terms, pattern length=186

Compiled 9 regex patterns.


## 4. Feature Computation Functions

In [ ]:
def count_matches(text, pattern):
    """Count non-overlapping word-boundary matches of pattern in text."""
    if pd.isna(text) or (isinstance(text, str) and text.strip() == ''):
        return 0
    return len(pattern.findall(str(text)))

def word_count(text):
    """Count words in text."""
    if pd.isna(text) or (isinstance(text, str) and text.strip() == ''):
        return 0
    return len(str(text).split())

# Quick test
test_text = df['review_text'].iloc[0]
print(f"Test text (first 200 chars): {test_text[:200]}...")
for cat_key in list(cat_keys)[:3]:
    c = count_matches(test_text, patterns[cat_key])
    print(f"  {cat_key}: {c} matches in test text")
print(f"  Word count: {word_count(test_text)}")

Test text (first 200 chars): It doesn't say it's a Bruichladdich but it cannot be Browmore, can it? Colour: white wine. Nose: the freshest freshness ever and a brininess that's even more obvious than usual. Much more obvious… So ...
  fruit_aromatics: 5 matches in test text
  peat_smoke_coastal: 5 matches in test text
  sherry_rancio_oxidative: 0 matches in test text
  Word count: 128


## 5. Compute Features for All Scopes and Categories

Loop over scopes and categories. Compute count, proportion, per-1k-tokens, and binary presence for each combination.

**Performance note:** With 11,149 rows × 6 scopes × 9 categories, this will run ~600k regex searches. Expect 2-5 minutes.

In [ ]:
print("Computing features...")
t0 = time.time()

# Initialize results dataframe with metadata columns
features = pd.DataFrame()
features['dedupe_hash'] = df['dedupe_hash']

# Track column order
feature_cols = []

for scope in SCOPES:
    print(f"\n  Scope: {scope}...")
    scope_t0 = time.time()

    # Get text series for this scope
    text_series = df[scope]

    # Word count for this scope
    wc_col = f'wordcount_{scope}'
    features[wc_col] = text_series.apply(word_count)
    feature_cols.append(wc_col)

    for cat_key in cat_keys:
        short = CAT_SHORT[cat_key]
        pattern = patterns[cat_key]

        # Count matches
        count_col = f'{short}_{scope}_count'
        features[count_col] = text_series.apply(lambda t: count_matches(t, pattern))
        feature_cols.append(count_col)

        # Proportion
        prop_col = f'{short}_{scope}_prop'
        features[prop_col] = np.where(
            features[wc_col] > 0,
            features[count_col] / features[wc_col],
            np.nan
        )
        feature_cols.append(prop_col)

        # Per 1k tokens
        per1k_col = f'{short}_{scope}_per1k'
        features[per1k_col] = np.where(
            features[wc_col] > 0,
            (features[count_col] / features[wc_col]) * 1000,
            np.nan
        )
        feature_cols.append(per1k_col)

        # Binary presence
        binary_col = f'{short}_{scope}_binary'
        features[binary_col] = (features[count_col] > 0).astype(int)
        feature_cols.append(binary_col)

    scope_elapsed = time.time() - scope_t0
    print(f"    Completed in {scope_elapsed:.1f}s")

# Compute derived features: total dictionary hits per scope
for scope in SCOPES:
    wc_col = f'wordcount_{scope}'

    # Total count across all categories
    total_col = f'dict_total_{scope}_count'
    cat_count_cols = [f'{CAT_SHORT[cat]}_{scope}_count' for cat in cat_keys]
    features[total_col] = features[cat_count_cols].sum(axis=1)
    feature_cols.append(total_col)

    # Total per 1k
    total_per1k_col = f'dict_total_{scope}_per1k'
    features[total_per1k_col] = np.where(
        features[wc_col] > 0,
        (features[total_col] / features[wc_col]) * 1000,
        np.nan
    )
    feature_cols.append(total_per1k_col)

elapsed = time.time() - t0
print(f"\nFeature computation complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"Total feature columns: {len(feature_cols)}")

Computing features...

  Scope: review_text...
    Completed in 9.2s

  Scope: nose...
    Completed in 2.9s

  Scope: mouth...
    Completed in 2.5s

  Scope: finish...
    Completed in 0.8s

  Scope: comments...
    Completed in 1.3s

  Scope: nmf...
    Completed in 6.2s

Feature computation complete in 22.9s (0.4 min)
Total feature columns: 234


## 6. Add Metadata Columns

Carry forward key columns from the analytical dataset for easy joining and direct use in modeling.

In [ ]:
metadata_cols = ['score', 'review_year', 'distillery', 'review_length',
                 'identity_status', 'match_source']

for col in metadata_cols:
    if col in df.columns:
        features[col] = df[col]
        feature_cols.append(col)
    else:
        print(f"  WARNING: {col} not in dataframe")

print(f"Added {len(metadata_cols)} metadata columns")
print(f"Final feature matrix: {len(features)} rows x {len(feature_cols)} columns")

Added 6 metadata columns
Final feature matrix: 11149 rows x 240 columns


## 7. Validation Checks

In [ ]:
print("=" * 60)
print("VALIDATION CHECKS")
print("=" * 60)

# 7a. Sanity check: manually verify counts for a few reviews
print("\n7a. Manual spot-check (first 3 reviews):")
for i in range(3):
    print(f"\n  Review {i} (score={df['score'].iloc[i]}, {df['distillery'].iloc[i]}):")
    text = df['review_text'].iloc[i]
    print(f"    Text length: {word_count(text)} words")
    for cat_key in list(cat_keys)[:4]:
        short = CAT_SHORT[cat_key]
        ct = features[f'{short}_review_text_count'].iloc[i]
        binary = features[f'{short}_review_text_binary'].iloc[i]
        if ct > 0:
            # Find which terms matched
            pattern = patterns[cat_key]
            matched = set()
            for m in pattern.finditer(str(text)):
                matched.add(m.group().lower())
            print(f"    {cat_key}: count={ct}, matched={sorted(matched)[:5]}...")

# 7b. Distribution summary: mean per1k for each category (review_text scope)
print("\n7b. Mean per-1k-tokens rate by category (review_text scope):")
for cat_key in cat_keys:
    short = CAT_SHORT[cat_key]
    col = f'{short}_review_text_per1k'
    mean_val = features[col].mean()
    std_val = features[col].std()
    print(f"  {cat_key:<30s}: mean={mean_val:6.2f}, std={std_val:6.2f}")

# 7c. Correlation with score
print("\n7c. Pearson correlation with score (review_text_per1k):")
for cat_key in cat_keys:
    short = CAT_SHORT[cat_key]
    col = f'{short}_review_text_per1k'
    corr = features[col].corr(features['score'])
    print(f"  {cat_key:<30s}: r={corr:+.4f}")

# 7d. Category coverage: % of reviews with >=1 hit
print("\n7d. Category coverage - % reviews with >=1 hit (review_text scope):")
for cat_key in cat_keys:
    short = CAT_SHORT[cat_key]
    col = f'{short}_review_text_binary'
    pct = features[col].mean() * 100
    print(f"  {cat_key:<30s}: {pct:5.1f}%")

# 7e. Cross-check: total count = sum of all category counts
print("\n7e. Cross-check: dict_total = sum of category counts")
cat_count_cols = [f'{CAT_SHORT[cat]}_review_text_count' for cat in cat_keys]
computed_total = features[cat_count_cols].sum(axis=1)
stored_total = features['dict_total_review_text_count']
max_diff = (computed_total - stored_total).abs().max()
print(f"  Max difference: {max_diff}")
assert max_diff == 0, "Total count mismatch!"
print("  OK - totals match exactly")

# 7f. Check for NaN patterns
print("\n7f. NaN check:")
for scope in SCOPES:
    wc = features[f'wordcount_{scope}']
    n_zero_wc = (wc == 0).sum()
    # Count NaNs in prop columns for this scope
    prop_cols = [f'{CAT_SHORT[cat]}_{scope}_prop' for cat in cat_keys]
    n_nan_props = features[prop_cols].isna().any(axis=1).sum()
    count_cols = [f'{CAT_SHORT[cat]}_{scope}_count' for cat in cat_keys]
    n_nan_counts = features[count_cols].isna().any(axis=1).sum()
    print(f"  {scope}: {n_zero_wc} zero-wordcount rows, {n_nan_props} rows with NaN props, {n_nan_counts} rows with NaN counts")

VALIDATION CHECKS

7a. Manual spot-check (first 3 reviews):

  Review 0 (score=87, Bruichladdich):
    Text length: 128 words
    fruit_aromatics: count=5, matched=['lemon', 'lime', 'peach']...
    peat_smoke_coastal: count=5, matched=['brine', 'oyster', 'salt', 'sea_air', 'seaweed']...
    oak_cask_wood: count=2, matched=['oak', 'vanilla']...

  Review 1 (score=89, Bruichladdich):
    Text length: 209 words
    fruit_aromatics: count=1, matched=['raspberry']...
    peat_smoke_coastal: count=4, matched=['bonfire', 'coal', 'peat']...
    sherry_rancio_oxidative: count=6, matched=['chestnut', 'coffee', 'espresso', 'leather', 'sherry']...
    oak_cask_wood: count=3, matched=['ex_sherry', 'pencil_shavings', 'wood']...

  Review 2 (score=94, Glenfiddich):
    Text length: 254 words
    fruit_aromatics: count=7, matched=['clove', 'dried_fruit', 'jam', 'mint', 'orange']...
    sherry_rancio_oxidative: count=6, matched=['balsamic', 'cedar', 'dark_chocolate', 'leather', 'polish']...
    oak_cas

## 8. Save Feature Matrix

In [ ]:
# Reorder columns: metadata first, then features
final_cols = ['dedupe_hash'] + metadata_cols + [c for c in feature_cols if c not in metadata_cols and c != 'dedupe_hash']
features = features[final_cols]

# Save
features.to_parquet('data/whiskyfun_dict_features.parquet', index=False)
print(f"Saved: data/whiskyfun_dict_features.parquet")
print(f"Shape: {features.shape[0]} rows x {features.shape[1]} columns")

# Quick reload verification
test = pd.read_parquet('data/whiskyfun_dict_features.parquet')
print(f"Reloaded: {test.shape[0]} rows x {test.shape[1]} columns")
assert test.shape[0] == 11149, "Row count mismatch!"
print("Verification OK")

# Print column list
print(f"\nAll columns ({len(test.columns)}):")
for c in test.columns:
    print(f"  {c}")

Saved: data/whiskyfun_dict_features.parquet
Shape: 11149 rows x 241 columns
Reloaded: 11149 rows x 241 columns
Verification OK

All columns (241):
  dedupe_hash
  score
  review_year
  distillery
  review_length
  identity_status
  match_source
  wordcount_review_text
  fruit_review_text_count
  fruit_review_text_prop
  fruit_review_text_per1k
  fruit_review_text_binary
  peat_review_text_count
  peat_review_text_prop
  peat_review_text_per1k
  peat_review_text_binary
  sherry_review_text_count
  sherry_review_text_prop
  sherry_review_text_per1k
  sherry_review_text_binary
  oak_review_text_count
  oak_review_text_prop
  oak_review_text_per1k
  oak_review_text_binary
  texture_review_text_count
  texture_review_text_prop
  texture_review_text_per1k
  texture_review_text_binary
  mineral_review_text_count
  mineral_review_text_prop
  mineral_review_text_per1k
  mineral_review_text_binary
  flaw_review_text_count
  flaw_review_text_prop
  flaw_review_text_per1k
  flaw_review_text_binary

## 9. Summary

### Feature Matrix Statistics
- Rows: 11,149 (one per review)
- Feature columns: 9 categories x 6 scopes x 4 types = 216 category features
- Plus: 12 derived features (dict_total_count, dict_total_per1k per scope)
- Plus: 6 wordcount columns
- Plus: 7 metadata columns (dedupe_hash, score, review_year, distillery, review_length, identity_status, match_source)
- Total: ~241 columns

### Next Steps
The feature matrix is ready for:
- `w1-eda` Sections 6-7 (dictionary-based EDA)
- `w2` regression models and known-groups validation